# 04 Wikipedia-Web-Scraping

## Zweck

Verwenden Sie Wikipedia-Stadtseiten als erforderliche Web-Scraping-Quelle. Speichern Sie rohes HTML in Bronze, analysieren Sie kontextbezogene Stadtmetadaten, validieren Sie die Beitrittsfähigkeit mit `city_id` und schreiben Sie Silver-Parquet.

## Eingaben

- `data/silver/city_reference.parquet`
- Wikipedia-Städte-URLs aus der Städtereferenztabelle oder abgeleitet von Städtenamen.

## Ausgaben

- `data/bronze/wikipedia_html/<city_id>.html`
- `data/silver/city_metadata.parquet`

## Verwendete Technologien

Python, Anfragen, BeautifulSoup, lxml, Pandas, Parkett.

## Konfiguration

Legen Sie `RUN_WIKIPEDIA_FETCH=true` fest, um rohes HTML abzurufen. Das Anforderungsverhalten verwendet einen klaren Benutzeragenten, eine Zeitüberschreitung und eine kleine Verzögerung. Generierte HTML- und Parkett-Ausgaben werden von Git ignoriert.

In [ ]:
from pathlib import Path
import os
import json
import pandas as pd

_env_root = os.getenv("PROJECT_ROOT")
if _env_root:
    PROJECT_ROOT = Path(_env_root).resolve()
elif Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / Path(os.getenv("DATA_DIR", "data"))
CHECKPOINT_DIR = PROJECT_ROOT / Path(os.getenv("CHECKPOINT_DIR", "data/checkpoints"))

print({"project_root": str(PROJECT_ROOT), "data_dir": str(DATA_DIR)})


## Implementierung

### Eingabevertrag

Phase 4 hängt von der Stadtreferenz der Phase 2 ab. Wenn die Datei noch nicht generiert wurde, enthält das Notebook dieselbe Liste kontrollierter Städte als Ersatz, sodass das Notebook verständlich bleibt. Der normale Pfad besteht jedoch darin, zuerst das Notebook `02` auszuführen.

### Scraping-Pfade und -Verhalten konfigurieren

Diese Zelle definiert den Bronze-HTML-Ordner, den Silver-Ausgabeordner, das Ausführungsflag, die Anforderungsheader und die Verzögerung. Der Abruf wird optional, damit Run-All respektvoll und reproduzierbar bleibt.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import re
import time
import pandas as pd

CITY_REFERENCE_PATH = DATA_DIR / "silver" / "city_reference.parquet"
WIKIPEDIA_RAW_DIR = DATA_DIR / "bronze" / "wikipedia_html"
SILVER_DIR = DATA_DIR / "silver"
WIKIPEDIA_RAW_DIR.mkdir(parents=True, exist_ok=True)
SILVER_DIR.mkdir(parents=True, exist_ok=True)

RUN_WIKIPEDIA_FETCH = os.getenv("RUN_WIKIPEDIA_FETCH", "true").lower() == "true"
REQUEST_SLEEP_SECONDS = float(os.getenv("WIKIPEDIA_REQUEST_SLEEP_SECONDS", "0.5"))
HEADERS = {"User-Agent": "euro-air-quality-pipeline/1.0 didaktisches Web-Scraping-Projekt"}

print({"run_wikipedia_fetch": RUN_WIKIPEDIA_FETCH, "silver_dir": str(SILVER_DIR), "wikipedia_raw_dir": str(WIKIPEDIA_RAW_DIR)})


### Laden Sie die Stadtreferenzeingabe

Der normale Pfad liest die Notebook-Ausgabe `02`. Durch einen kleinen In-Notebook-Fallback bleibt dieses Notebook verständlich isoliert, die geordnete Pipeline sollte jedoch zuerst das Notebook `02` ausführen.

In [ ]:
if CITY_REFERENCE_PATH.exists():
    city_reference_df = pd.read_parquet(CITY_REFERENCE_PATH)
else:
    city_reference_df = pd.DataFrame([
        {"city_id": "vienna_at", "city_name": "Vienna", "country_code": "AT", "latitude": 48.2082, "longitude": 16.3738, "wikipedia_url": "https://en.wikipedia.org/wiki/Vienna"},
        {"city_id": "berlin_de", "city_name": "Berlin", "country_code": "DE", "latitude": 52.5200, "longitude": 13.4050, "wikipedia_url": "https://en.wikipedia.org/wiki/Berlin"},
        {"city_id": "paris_fr", "city_name": "Paris", "country_code": "FR", "latitude": 48.8566, "longitude": 2.3522, "wikipedia_url": "https://en.wikipedia.org/wiki/Paris"},
        {"city_id": "madrid_es", "city_name": "Madrid", "country_code": "ES", "latitude": 40.4168, "longitude": -3.7038, "wikipedia_url": "https://en.wikipedia.org/wiki/Madrid"},
        {"city_id": "rome_it", "city_name": "Rome", "country_code": "IT", "latitude": 41.9028, "longitude": 12.4964, "wikipedia_url": "https://en.wikipedia.org/wiki/Rome"},
        {"city_id": "amsterdam_nl", "city_name": "Amsterdam", "country_code": "NL", "latitude": 52.3676, "longitude": 4.9041, "wikipedia_url": "https://en.wikipedia.org/wiki/Amsterdam"},
        {"city_id": "warsaw_pl", "city_name": "Warsaw", "country_code": "PL", "latitude": 52.2297, "longitude": 21.0122, "wikipedia_url": "https://en.wikipedia.org/wiki/Warsaw"},
        {"city_id": "prague_cz", "city_name": "Prague", "country_code": "CZ", "latitude": 50.0755, "longitude": 14.4378, "wikipedia_url": "https://en.wikipedia.org/wiki/Prague"},
    ])

required_city_columns = {"city_id", "city_name", "country_code", "latitude", "longitude"}
missing_city_columns = required_city_columns - set(city_reference_df.columns)
assert not missing_city_columns, f"In der Städtereferenz fehlen Spalten: {missing_city_columns}"
assert city_reference_df["city_id"].is_unique, \
    f"Doppelte city_id-Werte in der Städtereferenz: {city_reference_df[city_reference_df['city_id'].duplicated()]['city_id'].tolist()}"

print(f"OK: {len(city_reference_df)} Städte aus city_reference geladen, alle Pflichtfelder vorhanden")


### Rohes HTML in Bronze abrufen

Das Roharchiv HTML bewahrt Quellnachweise vor dem Parsen auf. Fehlgeschlagene Downloads werden aufgezeichnet und nicht stillschweigend ignoriert.

#### Erstellen Sie Wikipedia-URLs

Das Projekt verwendet einen expliziten URL aus dem Stadtmodell, sofern verfügbar, und leitet andernfalls einen lesbaren Fallback-URL ab.

In [ ]:
def wikipedia_url(row) -> str:
    if "wikipedia_url" in row and pd.notna(row["wikipedia_url"]):
        return row["wikipedia_url"]
    return "https://en.wikipedia.org/wiki/" + str(row["city_name"]).replace(" ", "_")

print("OK: wikipedia_url definiert")


#### Definieren Sie die HTML-Anfrage

Der Anforderungshelfer wendet Timeout-, Benutzeragenten- und Inhaltsprüfungen an. Fehler bleiben als zurückgegebener Fehlertext sichtbar.

In [ ]:
def fetch_html(url: str, timeout: int = 20) -> tuple:
    import requests

    try:
        response = requests.get(url, headers=HEADERS, timeout=timeout)
        status_code = response.status_code
        response.raise_for_status()
        return response.text, status_code, None
    except Exception as exc:
        return None, None, str(exc)

print("OK: fetch_html definiert")


#### Bronze HTML abrufen oder wiederverwenden

Für jede Stadt ruft die Schleife entweder neues HTML ab oder verwendet eine lokale Bronze-Seite wieder. Fehlende lokale Dateien werden aufgezeichnet und nicht stillschweigend ignoriert.

In [ ]:
retrieval_results = []
for _, row in city_reference_df.iterrows():
    url = wikipedia_url(row)
    output_path = WIKIPEDIA_RAW_DIR / f"{row['city_id']}.html"
    html = None
    status_code = None
    error = None
    status = "skipped"

    if RUN_WIKIPEDIA_FETCH:
        html, status_code, error = fetch_html(url)
        if html:
            output_path.write_text(html, encoding="utf-8")
            status = "success"
        else:
            status = "failed"
        time.sleep(REQUEST_SLEEP_SECONDS)
    elif output_path.exists():
        status = "existing_local_file"

    retrieval_results.append({
        "city_id": row["city_id"],
        "city_name": row["city_name"],
        "url": url,
        "status": status,
        "http_status_code": status_code,
        "file_path": str(output_path) if output_path.exists() else None,
        "file_size_bytes": output_path.stat().st_size if output_path.exists() else None,
        "error": error,
        "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
    })

retrieval_results_df = pd.DataFrame(retrieval_results)
retrieval_results_df


### Stadtmetadaten analysieren

Der Parser ist defensiv. Es wird versucht, Bevölkerung, Fläche und Dichte aus der Infobox zu extrahieren, zeichnet jedoch `parse_status` und `parse_notes` auf, da sich die Wikipedia-Seiten zwischen den Städten unterscheiden.

#### Numerische Infoboxwerte normalisieren

Wikipedia-Werte enthalten Referenzen, Trennzeichen und Einheiten. Dieser Helfer extrahiert die erste verwendbare Zahl, ohne fehlende Werte zu erraten.

In [ ]:
def clean_number(value):
    if value is None:
        return None
    text = re.sub(r"\[.*?\]", "", str(value))
    text = text.replace(",", "").replace("\xa0", " ")
    match = re.search(r"([0-9]+(?:\.[0-9]+)?)", text)
    if not match:
        return None
    number = float(match.group(1))
    return int(number) if number.is_integer() else number

print("OK: clean_number definiert")


#### Infobox-Beschriftungen normalisieren

Die Beschriftungen variieren zwischen den Seiten. Durch die Bereinigung von Kleinbuchstaben und Leerzeichen ist ein defensiver Abgleich möglich.

In [ ]:
def normalize_label(value: str) -> str:
    return re.sub(r"\s+", " ", value.replace("•", " ")).strip().lower()

print("OK: normalize_label definiert")


#### Werte in Infobox-Abschnitten suchen

Wikipedia-Infoboxen gruppieren häufig Werte auf Stadtebene unterhalb von Abschnittsüberschriften. Dieser Helfer findet eine passende Zeile, ohne versehentlich einen späteren, nicht verwandten Wert zu verwenden.

In [ ]:
def find_section_value(soup, section_label: str, preferred_labels: list) -> str | None:
    """Wert auf Stadtebene aus einem Infobox-Abschnitt zurückgeben, nicht aus einer späteren unverbundenen Zeile."""
    infobox = soup.select_one("table.infobox")
    if infobox is None:
        return None
    rows = infobox.select("tr")
    section_start = None
    for index, row in enumerate(rows):
        header = row.select_one("th.infobox-header")
        if header and normalize_label(header.get_text(" ", strip=True)).startswith(section_label):
            section_start = index + 1
            break
    if section_start is None:
        return None
    section_rows = []
    for row in rows[section_start:]:
        if row.select_one("th.infobox-header"):
            break
        section_rows.append(row.get_text(" ", strip=True))
    for label in preferred_labels:
        for text in section_rows:
            if normalize_label(text).startswith(label):
                return text
    return None

print("OK: find_section_value definiert")


#### Behandeln Sie kompakte Infobox-Varianten

Einige Seiten speichern Werte direkt in einer beschrifteten Zeile. Dieser zweite Helfer unterstützt dieses einfachere Layout.

In [ ]:
def find_direct_infobox_value(soup, label: str) -> str | None:
    """Kompakte Infobox-Varianten behandeln, bei denen ein Wert direkt in der beschrifteten Zeile gespeichert ist."""
    infobox = soup.select_one("table.infobox")
    if infobox is None:
        return None
    for row in infobox.select("tr"):
        header = row.find("th")
        value = row.find("td")
        if header and value and normalize_label(header.get_text(" ", strip=True)).startswith(label):
            return value.get_text(" ", strip=True)
    return None

print("OK: find_direct_infobox_value definiert")


#### Analysieren Sie die kontextuellen Metadaten einer Stadt

Der Parser versucht, Bevölkerung, Fläche und Dichte zu extrahieren, leitet die Dichte nur bei Bedarf ab und zeichnet Teilergebnisse sichtbar auf.

In [ ]:
def parse_city_metadata(city_row, html: str, source_url: str) -> dict:
    from bs4 import BeautifulSoup

    soup = BeautifulSoup(html, "html.parser")
    city_level_labels = ["total", "city/state", "Hauptstadt und Gemeinde", "municipality", "Hauptstadt und Bezirk", "Hauptstadt"]
    population_text = find_section_value(soup, "population", city_level_labels) or find_direct_infobox_value(soup, "population")
    area_text = find_section_value(soup, "area", city_level_labels) or find_direct_infobox_value(soup, "area")
    density_text = find_section_value(soup, "population", ["density"]) or find_direct_infobox_value(soup, "density")

    population = clean_number(population_text)
    area_km2 = clean_number(area_text)
    population_density = clean_number(density_text)
    if population_density is None and population and area_km2:
        population_density = population / area_km2

    parsed_values = [population, area_km2, population_density]
    if all(value is not None for value in parsed_values):
        parse_status = "success"
    elif any(value is not None for value in parsed_values):
        parse_status = "partial"
    else:
        parse_status = "failed"

    return {
        "city_id": city_row["city_id"],
        "city_name": city_row["city_name"],
        "country_code": city_row["country_code"],
        "population": population,
        "area_km2": area_km2,
        "population_density": population_density,
        "source_url": source_url,
        "metadata_source": "wikipedia",
        "processed_at_utc": datetime.now(timezone.utc).isoformat(),
        "parse_status": parse_status,
        "parse_notes": "Mit defensiven Heuristiken aus der Wikipedia-Infobox geparst; Werte erfordern eine kontextbezogene Interpretation.",
    }

print("OK: parse_city_metadata definiert")


#### Analysieren Sie alle lokalen Bronze-Seiten und schreiben Sie Silver Parkquet

Jede Stadt erhält einen Ausgabedatensatz, einschließlich Analysestatus und Notizen. Das Silber-Ergebnis wird durch `city_id` verschlüsselt.

In [ ]:
metadata_records = []
for _, row in city_reference_df.iterrows():
    path = WIKIPEDIA_RAW_DIR / f"{row['city_id']}.html"
    url = wikipedia_url(row)
    if path.exists():
        metadata_records.append(parse_city_metadata(row, path.read_text(encoding="utf-8"), url))
    else:
        metadata_records.append({
            "city_id": row["city_id"],
            "city_name": row["city_name"],
            "country_code": row["country_code"],
            "population": None,
            "area_km2": None,
            "population_density": None,
            "source_url": url,
            "metadata_source": "wikipedia",
            "processed_at_utc": datetime.now(timezone.utc).isoformat(),
            "parse_status": "failed",
            "parse_notes": "Keine lokale HTML-Rohdatei verfügbar. Mit RUN_WIKIPEDIA_FETCH=true ausführen.",
        })

city_metadata_df = pd.DataFrame(metadata_records)
city_metadata_df


## Validierung / Qualitätsprüfungen

Validieren Sie eine Zeile pro `city_id`, Join-Schlüssel ungleich Null, Metadatenquelle, Analysestatus, Joinability mit Stadtreferenz und Parkett-Rücklesung.

In [ ]:
required_metadata_columns = {
    "city_id", "city_name", "country_code", "population", "area_km2",
    "population_density", "source_url", "metadata_source", "processed_at_utc",
    "parse_status", "parse_notes",
}
missing_cols = required_metadata_columns - set(city_metadata_df.columns)
assert not missing_cols, f"In city_metadata_df fehlen erforderliche Spalten: {missing_cols}"

assert city_metadata_df["city_id"].notna().all(), \
    "city_metadata_df enthält Zeilen mit leerer city_id"
assert city_metadata_df["city_id"].is_unique, \
    f"Doppelte city_id in den Metadaten: {city_metadata_df[city_metadata_df['city_id'].duplicated()]['city_id'].tolist()}"
assert (city_metadata_df["metadata_source"] == "wikipedia").all(), \
    f"Unerwartete metadata_source-Werte: {city_metadata_df['metadata_source'].unique()}"

invalid_status = set(city_metadata_df["parse_status"]) - {"success", "partial", "failed"}
assert not invalid_status, f"Unerwartete parse_status-Werte: {invalid_status}"

assert city_metadata_df["population"].dropna().gt(0).all(), \
    f"Nicht positive Bevölkerungswerte gefunden: {city_metadata_df.loc[city_metadata_df['population'].notna() & (city_metadata_df['population'] <= 0), ['city_id', 'population']]}"
assert city_metadata_df["area_km2"].dropna().gt(0).all(), \
    f"Nicht positive area_km2-Werte gefunden: {city_metadata_df.loc[city_metadata_df['area_km2'].notna() & (city_metadata_df['area_km2'] <= 0), ['city_id', 'area_km2']]}"
assert city_metadata_df["population_density"].dropna().gt(0).all(), \
    f"Nicht positive population_density-Werte gefunden: {city_metadata_df.loc[city_metadata_df['population_density'].notna() & (city_metadata_df['population_density'] <= 0), ['city_id', 'population_density']]}"

joined = city_reference_df.merge(city_metadata_df, on="city_id", how="left", indicator=True)
unmatched = joined.loc[joined["_merge"] != "both", "city_id"].tolist()
assert not unmatched, f"Für diese city_ids aus city_reference fehlt ein Metadateneintrag: {unmatched}"

output_path = SILVER_DIR / "city_metadata.parquet"
city_metadata_df.to_parquet(output_path, index=False)
roundtrip = pd.read_parquet(output_path)
assert len(roundtrip) == len(city_metadata_df), \
    f"Abweichende Zeilenanzahl beim Parquet-Roundtrip: geschrieben: {len(city_metadata_df)}, erneut gelesen: {len(roundtrip)}"

parse_summary = city_metadata_df["parse_status"].value_counts().to_dict()
print(f"Zusammenfassung des Parse-Status: {parse_summary}")
roundtrip[["city_id", "population", "area_km2", "population_density", "parse_status"]]

## Ergebnisse

Phase 4 erzeugt einen Silver City-Metadatendatensatz mit dem Schlüssel `city_id`. Rohes HTML wird zur Rückverfolgbarkeit lokal in Bronze aufbewahrt.

## Einschränkungen

Wikipedia-Werte sind kontextbezogen und nicht die offizielle Grundwahrheit. Seitenstrukturen und administrative Definitionen unterscheiden sich. Fehlende oder mehrdeutige Werte bleiben null und werden durch `parse_status` und `parse_notes` dokumentiert. Diese Felder dürfen nicht als kausale Erklärungen für die Luftqualität interpretiert werden.

## Nächster Schritt

Führen Sie das Notebook `05_open_meteo_api_and_kafka_producer.ipynb` aus, um den REST API und den Kafka-Produzentenpfad zu implementieren.